## Creando un RAG

In [23]:
# Importando librerias necesarias
import os
from google import genai
from google.genai.types import (
 GenerateContentConfig,
 GoogleSearch,
 Tool)
from dotenv import load_dotenv
import time
import wsgiref
from google.genai import types
from pathlib import Path

In [24]:
# Cargando las claves API
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [25]:
# Definiendo el modelo
MODEL_ID = "gemini-2.5-flash-lite"

# Definiendo el RAG
FILE_PATH = "manual_tecnico.pdf"

# 1. Verificar si el archivo existe localmente antes de empezar
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"No se encontró el archivo: {FILE_PATH}")

In [26]:
# 2. Buscar si ya existe un store con ese nombre para no duplicar
existing_stores = client.file_search_stores.list()

file_search_store = next((s for s in existing_stores if s.display_name == 'mi_biblioteca_avanzada'), None)
if not file_search_store:
    print("Creando nuevo almacén...")
    file_search_store = client.file_search_stores.create(
    config={'display_name': 'mi_biblioteca_avanzada'})
else:
    print(f"Usando almacén existente: {file_search_store.name}")

Usando almacén existente: fileSearchStores/mibibliotecaavanzada-eyikgucx8xp5


In [27]:
# Subida e indexación del documento
print("Subiendo e indexando archivo...")

operation = client.file_search_stores.upload_to_file_search_store(
    file=FILE_PATH,
    file_search_store_name=file_search_store.name,
    config={'display_name': 'Manual de Usuario V1',
            'chunking_config': {
                'white_space_config': {
                    'max_tokens_per_chunk': 512,
                    'max_overlap_tokens': 50}
                }
            }
)

while not operation.done:
    print("Procesando...")
    time.sleep(5)
    operation = client.operations.get(operation)
print("¡Todo listo! Realizando consulta...")

Subiendo e indexando archivo...
Procesando...
¡Todo listo! Realizando consulta...


In [28]:
# 4. Consulta

response = client.models.generate_content(
    model=MODEL_ID,
    contents="¿Cómo se resetea el dispositivo según el manual?",
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=[file_search_store.name]
                )
            )
        ]
    )
)

print("\n" + "="*50)
print(f"RESPUESTA IA: {response.text}")
print("="*50 + "\n")


RESPUESTA IA: Para restablecer el dispositivo según el manual, tienes dos opciones principales:

1.  **Restablecer a la configuración de fábrica (borrando todos los ajustes):**
    *   Visita `http://tplinkwifi.net` e inicia sesión con tu ID de TP-Link o la contraseña que hayas configurado.
    *   Ve a `Avanzado > Herramientas del sistema > Copia de seguridad y restauración`.
    *   En la sección "Restauración de valores predeterminados de fábrica", haz clic en `RESTAURAR DE FÁBRICA`.
    *   Espera unos minutos para que el proceso de reinicio y restauración se complete. Durante este tiempo, no apagues ni reinicies el router. Se recomienda encarecidamente hacer una copia de seguridad de la configuración actual antes de proceder con el restablecimiento.

2.  **Restablecer a los valores predeterminados de fábrica (excepto la contraseña de inicio de sesión y el ID de TP-Link):**
    *   Visita `http://tplinkwifi.net` e inicia sesión con tu ID de TP-Link o la contraseña que hayas config

In [29]:
# 5. Mostrar fuentes (Citas)
if response.candidates[0].grounding_metadata:
    print("FUENTES CONSULTADAS:")
    for chunk in response.candidates[0].grounding_metadata.grounding_chunks:
        if chunk.retrieved_context:
            print(f"- {chunk.retrieved_context.text[:150]}...")

FUENTES CONSULTADAS:
- devices connected to this router via EasyMesh.
1. Visit http://tplinkwifi.net, and log in with your TP-Link ID or the password you set for
the router....
- satellite devices connected to this router via EasyMesh.
1. Visit http://tplinkwifi.net, and log in with your TP-Link ID or the password you set for
t...
- satellite devices connected to this router via EasyMesh.
1. Visit http://tplinkwifi.net, and log in with your TP-Link ID or the password you set for
t...
- satellite devices connected to this router via EasyMesh.
1. Visit http://tplinkwifi.net, and log in with your TP-Link ID or the password you set for
t...
- 1. 1.Auto Update

 | 
98



15. 1. 2.Online Upgrade.

 | 
98



15. 1. 3.Local Upgrade

 | 
99



15. 1. 4.EasyMesh Satellite Update.

 | 
100



15. ...
